# 🚀 Guía Maestra: Fine-Tuning de Modelos en Hugging Face (Plantilla Base)

Esta libreta es nuestra plantilla oficial para entrenar cualquier modelo de Inteligencia Artificial (TinyLlama, Qwen, Mistral, etc.).

Antes de ejecutar nada, recuerda los **4 Pilares (D.H.M.E.)** para que el sistema no colapse:
1. **(D) Datos limpios:** La IA aprende por repetición. Necesitamos formato estricto (Instrucción -> Respuesta).
2. **(H) Hardware compatible:** Obligaremos a la IA a hablar el idioma de nuestra GPU T4 (`torch.float16`) para evitar errores matemáticos.
3. **(M) Memoria bajo control:** Usaremos la técnica **LoRA** (parches de memoria) para no desbordar la RAM de la tarjeta gráfica.
4. **(E) Entrenamiento inteligente:** Ajustaremos las "palancas" (hiperparámetros) para decidir cómo estudia nuestra IA.

In [ ]:
# 1. Instalamos las herramientas del taller (Librerías de Hugging Face)
!pip install -q transformers datasets accelerate bitsandbytes peft trl

# 2. Importamos las piezas que vamos a usar
import torch
import gc
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 23.6 MB/s eta 0:00:00


## Paso 1: (D) Preparar los Datos (El Libro de Estudio)

Las IA no entienden tablas, entienden secuencias de texto. Aquí descargamos nuestro dataset y lo formateamos como si fueran **Tarjetas de Estudio** (*Flashcards*).
*Si cambiamos de proyecto (ej. a PySpark), solo tenemos que cambiar la función `formatear_prompt` para leer nuestro propio archivo JSON.*

In [ ]:
print("1. Cargando el Dataset (10,000 ejemplos)...")
dataset = load_dataset("b-mc2/sql-create-context", split="train[:10000]")

# Esta es la regla estricta de formato que la IA va a memorizar
def formatear_prompt(ejemplo):
    prompt = f"Pregunta: {ejemplo['question']}\nContexto: {ejemplo['context']}\nSQL: {ejemplo['answer']}"
    return {"texto_formateado": prompt}

dataset = dataset.map(formatear_prompt)
print("¡Datos listos y formateados!")

1. Cargando el Dataset (10,000 ejemplos)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sql_create_context_v4.json:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

¡Datos listos y formateados!


## Paso 2: Entrenando a TinyLlama (El Junior)

Aquí aplicamos los pilares **H** (Hardware) y **M** (Memoria).
1. Cargamos el cerebro base adaptándolo a `float16`.
2. Creamos `LoraConfig`: En lugar de modificar el cerebro entero (que haría explotar la GPU), le pegamos un "cuaderno de notas" de 50 MB llamado LoRA.


3. Configuramos `SFTConfig` (El Entrenamiento). Recuerda la metáfora del estudiante:
* `per_device_train_batch_size=4`: Pone 4 tarjetas sobre el pupitre a la vez.
* `num_train_epochs=1`: Lee el libro completo 1 sola vez (porque queremos que sea rápido).

In [ ]:
print("--- INICIANDO TINYLLAMA ---")
modelo_id_llama = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer_llama = AutoTokenizer.from_pretrained(modelo_id_llama)
tokenizer_llama.pad_token = tokenizer_llama.eos_token

# (H) Hardware compatible
modelo_llama = AutoModelForCausalLM.from_pretrained(
    modelo_id_llama, device_map="auto", torch_dtype=torch.float16
)

# (M) Memoria bajo control (LoRA)
lora_config_llama = LoraConfig(
    r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], bias="none", task_type="CAUSAL_LM"
)

# (E) Entrenamiento Inteligente
argumentos_llama = SFTConfig(
    output_dir="./modelo_llama_PRO",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=20,
    num_train_epochs=1, # Solo 1 pasada para ir rápido
    fp16=True, bf16=False, optim="adamw_torch", dataset_text_field="texto_formateado"
)

entrenador_llama = SFTTrainer(
    model=modelo_llama, train_dataset=dataset, peft_config=lora_config_llama,
    processing_class=tokenizer_llama, args=argumentos_llama
)

print("🚀 Entrenando TinyLlama...")
entrenador_llama.train()

print("💾 Guardando TinyLlama en disco virtual...")
entrenador_llama.model.save_pretrained("llama_sql_definitivo")

# LIMPIEZA VITAL: Borramos al estudiante de la memoria RAM para que entre el siguiente
del modelo_llama
del entrenador_llama
gc.collect()
torch.cuda.empty_cache()
print("🧹 Memoria liberada. ¡TinyLlama finalizado!")

--- INICIANDO TINYLLAMA ---


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Truncating train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


🚀 Entrenando TinyLlama...


Step,Training Loss
20,2.074788
40,1.282573
60,1.058188
80,1.017020
100,0.951568
120,0.988773
140,0.985183
160,0.963079
180,0.959085
200,0.948561


💾 Guardando TinyLlama en disco virtual...
🧹 Memoria liberada. ¡TinyLlama finalizado!


## Paso 3: Entrenando a Qwen (El Peso Pesado)

Qwen es más inteligente pero más grande. Para proteger la memoria de Colab, ajustamos nuestras palancas:
* Bajamos el pupitre a `batch_size=2` para que no explote.
* Aumentamos el borrador temporal `gradient_accumulation_steps=8` para compensar.
* Le exigimos más dándole `epochs=2` (2 pasadas completas).

In [ ]:
print("--- INICIANDO QWEN ---")
modelo_id_qwen = "Qwen/Qwen2.5-Coder-1.5B"
tokenizer_qwen = AutoTokenizer.from_pretrained(modelo_id_qwen)
tokenizer_qwen.pad_token = tokenizer_qwen.eos_token

modelo_qwen = AutoModelForCausalLM.from_pretrained(
    modelo_id_qwen, device_map="auto", torch_dtype=torch.float16
)

# Rango LoRA mayor (16) porque Qwen tiene más capacidad de absorber conocimiento
lora_config_qwen = LoraConfig(
    r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], bias="none", task_type="CAUSAL_LM"
)

argumentos_qwen = SFTConfig(
    output_dir="./modelo_qwen_HEAVY",
    per_device_train_batch_size=2, # Bajamos el batch por seguridad
    gradient_accumulation_steps=8, # Aumentamos la acumulación
    learning_rate=1e-4,
    logging_steps=50,
    num_train_epochs=2, # 2 Vueltas completas
    fp16=True, bf16=False, optim="adamw_torch", dataset_text_field="texto_formateado"
)

entrenador_qwen = SFTTrainer(
    model=modelo_qwen, train_dataset=dataset, peft_config=lora_config_qwen,
    processing_class=tokenizer_qwen, args=argumentos_qwen
)

print("🚀 Entrenando Qwen intensivamente...")
entrenador_qwen.train()

print("💾 Guardando Qwen en disco virtual...")
entrenador_qwen.model.save_pretrained("qwen_sql_definitivo")

# Limpieza total
del modelo_qwen
del entrenador_qwen
gc.collect()
torch.cuda.empty_cache()
print("🧹 Memoria liberada. ¡Qwen finalizado!")

--- INICIANDO QWEN ---


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Truncating train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


🚀 Entrenando Qwen intensivamente...


Step,Training Loss
50,1.663100
100,1.133212
150,1.103730
200,1.073812
250,1.047305
300,1.028089
350,1.020724
400,1.017895
450,1.018987
500,1.004723


💾 Guardando Qwen en disco virtual...
🧹 Memoria liberada. ¡Qwen finalizado!


## Paso 4: El Duelo Final (Inferencia / A-B Testing)

Una vez entrenados, la RAM está limpia. Ahora cargamos los modelos base "vírgenes", les enchufamos nuestros cuadernos de notas LoRA (los archivos `_definitivo` que guardamos) y les hacemos una pregunta trampa con baja `temperature` (para que sean exactos, no creativos).

In [ ]:
def probar_modelo(nombre_carpeta, modelo_base_id, pregunta, contexto):
    print(f"\n--- Probando: {nombre_carpeta} ---")

    tokenizer = AutoTokenizer.from_pretrained(modelo_base_id)
    tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        modelo_base_id, torch_dtype=torch.float16, device_map="auto"
    )
    # ¡Aquí ocurre la magia! Enganchamos tu LoRA al modelo
    model = PeftModel.from_pretrained(base_model, nombre_carpeta)

    prompt = f"Pregunta: {pregunta}\nContexto: {contexto}\nSQL: "
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=64, temperature=0.01, do_sample=False,
            eos_token_id=tokenizer.eos_token_id
        )

    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

    # Limpiar para el siguiente turno
    del model, base_model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

# Ejecutamos el duelo
pregunta = "¿Cuál es el nombre del empleado con el salario más alto en el departamento de 'Ventas'?"
contexto = "CREATE TABLE staff (nombre VARCHAR, salario INT, departamento VARCHAR)"

probar_modelo("llama_sql_definitivo", "TinyLlama/TinyLlama-1.1B-Chat-v1.0", pregunta, contexto)
probar_modelo("qwen_sql_definitivo", "Qwen/Qwen2.5-Coder-1.5B", pregunta, contexto)


--- Probando: llama_sql_definitivo ---


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Pregunta: ¿Cuál es el nombre del empleado con el salario más alto en el departamento de 'Ventas'?
Contexto: CREATE TABLE staff (nombre VARCHAR, salario INT, departamento VARCHAR)
SQL:  SELECT T1.nombre FROM staff AS T1 JOIN staff AS T2 ON T1.departamento = T2.departamento WHERE T2.salario > 10000000000000000000000000

--- Probando: qwen_sql_definitivo ---


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Pregunta: ¿Cuál es el nombre del empleado con el salario más alto en el departamento de 'Ventas'?
Contexto: CREATE TABLE staff (nombre VARCHAR, salario INT, departamento VARCHAR)
SQL:  SELECT nombre FROM staff WHERE salario = (SELECT MAX(salario) FROM staff WHERE departamento = 'Ventas') AND departamento = 'Ventas'<|fim_middle|>



## Paso 5: Guardado Seguro (Protección de nuestro trabajo)

Como Google Colab borra sus máquinas virtuales al cerrarse, tenemos que exportar nuestro trabajo ganador a nuestro Google Drive para no perder el LoRA.

In [ ]:
from google.colab import drive
import shutil

print("1. Conectando a Google Drive...")
drive.mount('/content/drive')

# 2. Copiamos la carpeta del ganador (Qwen) a tu Google Drive
# Asegúrate de crear una carpeta llamada "MisModelosIA" en tu Drive primero, o cambia la ruta.
origen = "/content/qwen_sql_definitivo"
destino = "/content/drive/MyDrive/modelo_qwen_sql_final"

print(f"2. Copiando el modelo a {destino}...")
shutil.copytree(origen, destino, dirs_exist_ok=True)
print("✅ ¡Modelo guardado de forma permanente en la nube!")

1. Conectando a Google Drive...
Mounted at /content/drive
2. Copiando el modelo a /content/drive/MyDrive/modelo_qwen_sql_final...
✅ ¡Modelo guardado de forma permanente en la nube!
